This notebook loads and displays the results of the GEE analyses.

In [14]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [15]:
import glob
import pickle
from prettytable import PrettyTable
from src.stat_utils import *

results_dir = os.path.join(parent_dir, 'results/statistics')
first_model_file = os.path.join(results_dir, 'gee_first.pkl')
full_model_file = os.path.join(results_dir, 'gee_full.pkl')

In [23]:
def add_row_to_table(session, results, table):
    corrected_vals = extract_pvals(results)
    predictors = corrected_vals.keys()
    for i, (predictor, pvalue) in enumerate(corrected_vals.items()):
        coefficient = results.params[predictor]
        confidence_interval = [results.conf_int()[0][predictor], results.conf_int()[1][predictor]]
        standard_error = results.standard_errors()[i]
        wald = results.wald_test(predictor, scalar=True)
        chi = wald.statistic
        pvalue = pvalue
        table.add_row([session, predictor, coefficient, confidence_interval, standard_error, chi, pvalue])

    return table

### Results of first session model

In [17]:
with open(first_model_file, 'rb') as f:
    gee_first = pickle.load(f)

print(gee_first.summary())

                               GEE Regression Results                              
Dep. Variable:                     Correct   No. Observations:                 6000
Model:                                 GEE   No. clusters:                        8
Method:                        Generalized   Min. cluster size:                 750
                      Estimating Equations   Max. cluster size:                 750
Family:                           Binomial   Mean cluster size:               750.0
Dependence structure:         Exchangeable   Num. iterations:                     6
Date:                     Tue, 05 Aug 2025   Scale:                           1.000
Covariance type:                    robust   Time:                         15:13:52
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
Intercept                         

In [ ]:
# Print the Wald Chi-Square statistics (these are adjusted for multiple comparisons)
print_wald_chi_square(gee_first)

Wald Chi-Square:
+--------------------------------------+--------------------+-----------------------+
|               Variable               |     Chi-Square     |        p-value        |
+--------------------------------------+--------------------+-----------------------+
|        ContrastHeterogeneity         | 18.361082803614064 | 5.482584808226089e-05 |
|            GridCoarseness            | 17.777698465296414 | 5.482584808226089e-05 |
| ContrastHeterogeneity:GridCoarseness | 15.26688399705535  | 9.333864182752107e-05 |
+--------------------------------------+--------------------+-----------------------+


### Results of full model

In [19]:
with open(full_model_file, 'rb') as f:
    gee_full = pickle.load(f)

print(gee_full.summary())

                               GEE Regression Results                              
Dep. Variable:                     Correct   No. Observations:                48000
Model:                                 GEE   No. clusters:                        8
Method:                        Generalized   Min. cluster size:                6000
                      Estimating Equations   Max. cluster size:                6000
Family:                           Binomial   Mean cluster size:              6000.0
Dependence structure:         Exchangeable   Num. iterations:                    15
Date:                     Tue, 05 Aug 2025   Scale:                           1.000
Covariance type:                    robust   Time:                         15:15:20
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
Intercept                         

In [26]:
# Print the Wald Chi-Square statistics (these are adjusted for multiple comparisons)
print_wald_chi_square(gee_full)

Wald Chi-Square:
+--------------------------------------+--------------------+-----------------------+
|               Variable               |     Chi-Square     |        p-value        |
+--------------------------------------+--------------------+-----------------------+
|        ContrastHeterogeneity         | 150.17554736209928 | 7.935227925304243e-34 |
|            GridCoarseness            | 127.6901266521975  | 5.248385018683855e-29 |
|              SessionID               | 29.911719884031275 | 9.043427567878702e-08 |
|   SessionID:ContrastHeterogeneity    | 71.56806961754121  | 8.035690406755229e-17 |
|       SessionID:GridCoarseness       | 1.4108291925000898 |   0.2349187913464244  |
| ContrastHeterogeneity:GridCoarseness | 169.2756434277779  | 6.390375383870478e-38 |
+--------------------------------------+--------------------+-----------------------+


### Simple effects for significant session interactions

In [25]:
table = PrettyTable()
table.field_names = ['session', 'predictor', 'beta-coefficient', '95% confidence interval', 'sdt error', 'Wald chi-square', 'P-value']

files = glob.glob(os.path.join(results_dir, 'gee_*.pkl'))
files.remove(full_model_file)
files.remove(first_model_file)
files = sorted(files)

for i, file in enumerate(sorted(files)):
    with open(file, 'rb') as f:
        results = pickle.load(f)
    table = add_row_to_table(i + 1, results, table)

# Print the table with results corrected for multiple comparisons
print(table)

+---------+-----------------------+---------------------+--------------------------------------------+---------------------+--------------------+------------------------+
| session |       predictor       |   beta-coefficient  |          95% confidence interval           |      sdt error      |  Wald chi-square   |        P-value         |
+---------+-----------------------+---------------------+--------------------------------------------+---------------------+--------------------+------------------------+
|    1    | ContrastHeterogeneity | -1.4584335093863061 | [-2.019373649151089, -0.8974933696215229]  | 0.26717882976035756 | 25.96787510849494  | 3.471461615896075e-07  |
|    2    | ContrastHeterogeneity |  -2.004631368760057 | [-2.7148236704546744, -1.2944390670654395] | 0.30268997779460965 | 30.606474383917227 | 3.1603737383895106e-08 |
|    3    | ContrastHeterogeneity | -2.3164232974684107 | [-2.9113408033401633, -1.7215057915966578] |  0.2960573087325982 | 58.23962009879802  |